# 🧭 02 · Geometry of molecules: coordinates, rotations, symmetry and invariance
### Turn atoms and density maps in 3D, measure how different two orientations are, find a symmetry axis, and build features that ignore orientation

**BiologyToResearch** · Part A — Foundations · Module 02 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/02_rotations_geometry.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`ndimage`, `special.sph_harm_y`), Matplotlib |
| **Prerequisites** | [Module 01](01_python_arrays_density_maps.ipynb) (arrays, `[z, y, x]`, maps from atoms). High-school trigonometry (sin, cos). |
| **Estimated time** | 90–120 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Xu, Zhang & Alber 2009 (rotation-invariant features — main); rotation-error metrics of Xu, Beck & Alber 2012, Gum-Net 2020 and BOE-ViT 2025; a rotation pitfall in the DISCA 2023 code — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments in this course are small, controlled *toy versions* of published methods. They teach the core idea of
> each paper; they do **not** reproduce the papers' numbers. **Nothing is stored in the repository:** molecules are
> real Protein Data Bank (PDB) models, downloaded the first time a notebook needs them and cached on your computer
> (offline, code-built stand-ins of the same size and symmetry are used instead); all images and data are simulated by
> code.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Compute** distances, angles and centres from atomic coordinates, and **check** them against known protein geometry.
2. **Derive** the 2D rotation matrix from the unit circle and **verify** its properties ($R^\top R = I$, $\det R = +1$).
3. **Build** 3D rotations (about x, y, z, about any axis, from ZYZ Euler angles), **show** that their order matters, and
   **explain** gimbal lock.
4. **Sample** uniformly random orientations correctly — and **demonstrate** why random Euler angles are not uniform.
5. **Rotate** a density map with interpolation, **superpose** two protein copies (Kabsch), and **detect** a reflection.
6. **Measure** rotation error both as a Frobenius distance and as a geodesic angle, and **convert** between them.
7. **Find** the 7-fold symmetry axis of the 20S proteasome by rotating and correlating.
8. **Implement** rotation-invariant spherical-harmonic shell features (the idea of Xu, Zhang & Alber 2009), **verify**
   their invariance, and **use** them to tell molecules apart at several noise levels.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, data | 5 min |
| 1 · Vectors and coordinates | distances, angles and centres in a real protein | 10 min |
| 2 · Rotations in 2D | derive the rotation matrix from the unit circle | 10 min |
| 3 · Rotations in 3D | x/y/z rotations, any axis, order matters | 10 min |
| 4 · Euler angles | the ZYZ convention, gimbal lock | 10 min |
| 5 · Random orientations | uniform sampling; a reflection pitfall | 10 min |
| 6 · Rotating a map | interpolation and its blur | 10 min |
| 7 · Rigid transforms | rotation + translation; superposing two subunits | 10 min |
| 8 · Measuring rotation error | Frobenius vs geodesic; how papers report errors | 10 min |
| 9 · Symmetry | find the proteasome's 7-fold axis | 10 min |
| 10 · Rotation-invariant features | spherical harmonics on shells (Xu, Zhang & Alber 2009) | 20 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

Inside a cell, every ribosome, proteasome and RuBisCO points in its own random direction — molecules tumble freely
before they are frozen. A tomogram therefore contains thousands of copies of the same machine, each **rotated** and
**shifted** differently. To average them into a sharp structure (Module 14), to recognise them (Modules 11, 16), or to
discover new ones (Module 15), we must constantly answer geometric questions: *How do I turn copy A onto copy B? How far
apart are two orientations? Which turns leave a symmetric molecule unchanged? Can I describe a molecule by numbers that
do not depend on how it is turned?* This module gives you the mathematics of turning — rotations — with real molecules.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook only needs a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed (not needed in this notebook).")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import scipy
from scipy import ndimage

SEED = 0
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| SciPy", scipy.__version__, "| seed", SEED)

### 🧰 Course toolkit

The helpers this notebook uses, copied from [`course_toolkit.py`](../course_toolkit.py): building and preparing
maps and atomic models, a backbone builder (`build_backbone`), the course's rotation functions (`rot_x/y/z`, `euler_zyz_to_matrix`, `matrix_to_euler_zyz`,
`random_rotation`, `rotation_angle_deg`, `rotate_volume`, `rotate_image`) and `pearson`/`normalize`. Every rotation
function is explained in this module — this is where they are taught.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v3.0; copied so this notebook runs alone) ──
import os
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_EXTRA_ENTRIES = {"5a1a": dict(name="β-galactosidase", organism="E. coli", mass_kDa=465, symmetry="D2")}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def atoms_to_pdb_lines(atoms, title="code-generated model (BiologyToResearch course)") -> list:
    """Format an atom dict (xyz, name, element, resname, chain, resnum) as PDB ATOM records (fixed columns)."""
    lines = [f"HEADER    {title[:40]:40s}", "REMARK   1 IDEALISED HELICES BUILT BY CODE - NOT AN EXPERIMENTAL STRUCTURE"]
    for i, (p, an, el, rn, ch, ri) in enumerate(zip(atoms["xyz"], atoms["name"], atoms["element"], atoms["resname"],
                                                    atoms["chain"], atoms["resnum"])):
        name = f" {an:<3s}" if len(an) < 4 else an
        lines.append(f"ATOM  {(i + 1) % 100000:5d} {name:4s} {rn:3s} {ch:1s}{int(ri) % 10000:4d}    "
                     f"{p[0]:8.3f}{p[1]:8.3f}{p[2]:8.3f}  1.00 30.00          {el:>2s}")
    lines.append("END")
    return lines


def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
def standin_atoms(pid):
    """Atomic model (dict of arrays) of the code-built stand-in for zoo entry `pid` (Å, centred at the origin).

    Used when the real entry cannot be downloaded: ideal α-helices arranged with the size, mass and symmetry of the
    PDB entry. Deterministic (the random numbers are seeded by the entry's name)."""
    pid = pid.lower()
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    return atoms


def pdb_cache_dir():
    """Folder (outside the repository) where downloaded PDB files are kept."""
    return os.environ.get("BIO2RESEARCH_PDB_CACHE") or os.path.join(os.path.expanduser("~"), ".cache",
                                                                    "biology_to_research", "pdb")


def _pdb_urls(pid, kind):
    """Download addresses of one PDB file: kind = 'entry' or 'assembly1' (mmCIF) or 'pdb' (legacy PDB format)."""
    pid = pid.lower()
    mid = pid[1:3]
    if kind == "entry":
        return [f"https://files.rcsb.org/download/{pid.upper()}.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/mmCIF/{mid}/{pid}.cif.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/{pid}.cif"]
    if kind == "assembly1":
        return [f"https://files.rcsb.org/download/{pid.upper()}-assembly1.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/assemblies/mmCIF/divided/{mid}/{pid}-assembly1.cif.gz"]
    if kind == "pdb":
        return [f"https://files.rcsb.org/download/{pid.upper()}.pdb.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/pdb/{mid}/pdb{pid}.ent.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/pdb{pid}.ent"]
    raise ValueError(kind)


def fetch_pdb(pid, kind="entry", timeout=60):
    """Text of a PDB file, downloaded once and cached outside the repository. Returns (text, where_from), or
    (None, reason) when no mirror can be reached (or BIO2RESEARCH_STANDINS is set)."""
    import gzip
    import urllib.request
    name = {"entry": f"{pid.lower()}.cif.gz", "assembly1": f"{pid.lower()}-assembly1.cif.gz",
            "pdb": f"{pid.lower()}.pdb.gz"}[kind]
    path = os.path.join(pdb_cache_dir(), name)
    if os.path.exists(path):
        return gzip.decompress(open(path, "rb").read()).decode("utf-8", "replace"), f"cached copy ({path})"
    if os.environ.get("BIO2RESEARCH_STANDINS"):
        return None, "BIO2RESEARCH_STANDINS is set"
    reason = "no mirror tried"
    for url in _pdb_urls(pid, kind):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                raw = response.read()
            text = (gzip.decompress(raw) if raw[:2] == b"\x1f\x8b" else raw).decode("utf-8", "replace")
            if ("_atom_site." not in text) and ("\nATOM  " not in text):
                raise ValueError("no atoms in the downloaded file")
            os.makedirs(pdb_cache_dir(), exist_ok=True)
            with open(path, "wb") as f:
                f.write(gzip.compress(text.encode()))
            return text, url
        except Exception as e:  # noqa: BLE001 — any failure: try the next mirror
            reason = f"{type(e).__name__}: {str(e)[:120]}"
    return None, reason


def _cif_tokens(line):
    """Split one line of an mmCIF loop into values (quoted values may contain spaces)."""
    if "'" not in line and '"' not in line:
        return line.split()
    import re
    toks = re.findall(r"'(?:[^']|'(?=\S))*'|\"(?:[^\"]|\"(?=\S))*\"|\S+", line)
    return [t[1:-1] if len(t) > 1 and t[0] == t[-1] and t[0] in "'\"" else t for t in toks]


_WATER = {"HOH", "DOD", "WAT", "H2O"}
def read_mmcif_atoms(text):
    """Atoms of the first model in an mmCIF file → dict(xyz, name, element, resname, chain, resnum).

    Reads the `_atom_site` loop; skips water and hydrogen, and keeps only the first of alternative atom locations."""
    lines = text.splitlines()
    i = next(k for k, line in enumerate(lines) if line.startswith("_atom_site."))
    cols = []
    while i < len(lines) and lines[i].startswith("_atom_site."):
        cols.append(lines[i].split()[0][len("_atom_site."):])
        i += 1
    c = {name: k for k, name in enumerate(cols)}
    get = lambda *names: next((c[n] for n in names if n in c), None)   # noqa: E731
    ix, iy, iz = c["Cartn_x"], c["Cartn_y"], c["Cartn_z"]
    i_el, i_name = get("type_symbol"), get("auth_atom_id", "label_atom_id")
    i_res, i_ch = get("auth_comp_id", "label_comp_id"), get("auth_asym_id", "label_asym_id")
    i_num, i_alt, i_mod = get("auth_seq_id", "label_seq_id"), get("label_alt_id"), get("pdbx_PDB_model_num")
    xyz, name, element, resname, chain, resnum = [], [], [], [], [], []
    first_model, buf = None, []
    for line in lines[i:]:
        if not line or line.startswith(("#", "_", "loop_", "data_")):
            break
        buf += _cif_tokens(line)
        while len(buf) >= len(cols):
            row, buf = buf[:len(cols)], buf[len(cols):]
            if i_mod is not None:
                first_model = row[i_mod] if first_model is None else first_model
                if row[i_mod] != first_model:
                    continue
            el = row[i_el].upper() if i_el is not None else row[i_name][:1].upper()
            if el in ("H", "D") or row[i_res] in _WATER:
                continue
            if i_alt is not None and row[i_alt] not in (".", "?", "A", "1"):
                continue
            xyz.append((float(row[ix]), float(row[iy]), float(row[iz])))
            name.append(row[i_name])
            element.append(el.capitalize() if len(el) > 1 else el)
            resname.append(row[i_res])
            chain.append(row[i_ch])
            num = row[i_num] if i_num is not None else "0"
            resnum.append(int(num) if num.lstrip("-").isdigit() else 0)
    return dict(xyz=np.array(xyz, np.float32), name=np.array(name), element=np.array(element),
                resname=np.array(resname), chain=np.array(chain), resnum=np.array(resnum))


_ATOMIC_MASS = {"H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999, "P": 30.974, "S": 32.06, "Se": 78.97, "Mg": 24.305,
                "Zn": 65.38, "Fe": 55.845, "Mn": 54.938, "K": 39.098, "Na": 22.99, "Cl": 35.45, "Ca": 40.078}
def model_mass_kDa(atoms):
    """Mass of an atomic model without hydrogens, plus ~8 % for the missing hydrogens (kDa)."""
    return 1.08 * sum(_ATOMIC_MASS.get(e, 12.0) for e in atoms["element"]) / 1000.0


def canonical_frame(atoms):
    """Centre a model and rotate it onto its principal axes — longest extent along x, shortest along z — with the
    signs fixed by the skewness of the atom distribution. A proteasome barrel then lies along x and a ring
    (RuBisCO, p97) lies in the x–y plane, like the stand-ins."""
    xyz = atoms["xyz"].astype(np.float64)
    X = xyz - xyz.mean(0)
    evecs = np.linalg.eigh(np.cov(X.T))[1][:, ::-1]                 # columns: principal axes, largest first
    Y = X @ evecs
    sign = np.sign(np.sum(Y ** 3, axis=0))
    sign[sign == 0] = 1.0
    Y *= sign
    if np.linalg.det(evecs * sign) < 0:                             # keep a proper rotation (no mirror image)
        Y[:, 2] *= -1
    out = dict(atoms)
    out["xyz"] = Y.astype(np.float32)
    return out


def pdb_atoms(pid, verbose=False):
    """Real atomic model of a PDB entry, in the canonical frame. Returns (atoms, description), or (None, reason).

    Uses the deposited entry unless its mass is far from ZOO's reference mass, in which case the first biological
    assembly is tried and the closer of the two kept (e.g. one ribosome rather than the two of a crystal)."""
    pid = pid.lower()
    ref = ZOO.get(pid, {}).get("mass_kDa") or _EXTRA_ENTRIES.get(pid, {}).get("mass_kDa")
    candidates, reason = [], ""
    for kind in ("entry", "assembly1"):
        if candidates and (ref is None or abs(np.log(model_mass_kDa(candidates[0][0]) / ref)) < np.log(1.3)):
            break
        text, where = fetch_pdb(pid, kind)
        if text is None:
            reason = where
            continue
        try:
            atoms = read_mmcif_atoms(text)
        except (StopIteration, KeyError, ValueError) as e:
            reason = f"could not read the file ({type(e).__name__})"
            continue
        if len(atoms["xyz"]):
            label = "deposited entry" if kind == "entry" else "biological assembly 1"
            candidates.append((atoms, f"PDB {pid.upper()} ({label}; {len(atoms['xyz']):,} atoms; from {where})"))
    if not candidates:
        return None, reason
    atoms, desc = min(candidates, key=lambda a: abs(np.log(model_mass_kDa(a[0]) / ref)) if ref else 0.0)
    if verbose:
        print(desc)
    return canonical_frame(atoms), desc


_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays, Å, centred) of zoo entry `pid`: the real PDB model when it can be downloaded, the
    code-built stand-in otherwise. The source is recorded in `atoms_source(pid)`."""
    pid = pid.lower()
    if pid not in _ATOMS_CACHE:
        atoms, desc = pdb_atoms(pid)
        if atoms is None:
            atoms, desc = standin_atoms(pid), f"code-built stand-in modelled on PDB {pid.upper()} (download failed: {desc})"
        _ATOMS_CACHE[pid] = (atoms, desc)
    return _ATOMS_CACHE[pid][0]


def pdb_text_lines(pid):
    """Lines of the legacy-format PDB file of an entry (downloaded once, cached) and their source. Offline, the lines
    are written by code for the stand-in (`atoms_to_pdb_lines`), in the same fixed-column format."""
    text, where = fetch_pdb(pid, "pdb")
    if text is not None:
        return text.splitlines(), f"PDB {pid.upper()} (legacy PDB format, from {where})"
    lines = atoms_to_pdb_lines(standin_atoms(pid), title=f"stand-in for PDB {pid.upper()}")
    return lines, f"code-built stand-in modelled on PDB {pid.upper()} (download failed: {where})"


def atoms_source(pid):
    """Where the atoms of zoo entry `pid` came from (a real PDB file or the offline stand-in)."""
    zoo_atoms(pid)
    return _ATOMS_CACHE[pid.lower()][1]


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0, "Mg": 12.0, "Zn": 30.0, "Fe": 26.0, "K": 19.0,
      "Na": 11.0, "Cl": 17.0, "Ca": 20.0, "Mn": 25.0, "Se": 34.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of a zoo entry, painted from its atoms (`zoo_atoms`)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, atoms_source(pid)


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def load_zoo(ids=("4v4r", "3j9i", "6utj", "1bxn"), size: int = 32, sigma: float = 1.0,
             verbose: bool = True) -> dict:
    """Build and prepare several zoo maps. Returns {pdb_id: volume(size³)}."""
    zoo = {}
    for pid in ids:
        raw, vs, src = zoo_map(pid)
        zoo[pid] = prepare_map(raw, size=size, sigma=sigma)
        if verbose:
            print(f"  {pid.upper():5s} {ZOO.get(pid, {}).get('name', ''):28s} raw {raw.shape} @ {vs:.0f} Å/voxel  ← {src}")
    return zoo


def rot_x(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])


def rot_y(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def rot_z(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def euler_zyz_to_matrix(phi, theta, psi):
    """R = Rz(phi) · Ry(theta) · Rz(psi)  (radians, acting on (x, y, z) vectors)."""
    return rot_z(phi) @ rot_y(theta) @ rot_z(psi)


def matrix_to_euler_zyz(R):
    """Inverse of :func:`euler_zyz_to_matrix` (one of the two equivalent solutions)."""
    theta = np.arccos(np.clip(R[2, 2], -1, 1))
    if np.isclose(np.sin(theta), 0):
        return np.arctan2(R[1, 0], R[0, 0]), theta, 0.0
    return np.arctan2(R[1, 2], R[0, 2]), theta, np.arctan2(R[2, 1], -R[2, 0])


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


def rotation_angle_deg(R1, R2) -> float:
    """Geodesic distance between two rotations: arccos((tr(R1ᵀR2) − 1)/2), in degrees."""
    c = (np.trace(R1.T @ R2) - 1) / 2
    return float(np.degrees(np.arccos(np.clip(c, -1, 1))))


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def rotate_image(img: np.ndarray, angle_rad: float, shift_xy=(0, 0), order: int = 1) -> np.ndarray:
    """2D analogue of :func:`rotate_volume` (counter-clockwise in the (x, y) plane)."""
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    R = np.array([[c, -s], [s, c]])
    R_idx = R[::-1, ::-1]                      # (x,y) -> (y,x)
    ctr = (np.array(img.shape) - 1) / 2.0
    t_idx = np.asarray(shift_xy, float)[::-1]
    M = R_idx.T
    return ndimage.affine_transform(img, M, offset=ctr - M @ (ctr + t_idx), order=order, mode="constant")


def normalize(v: np.ndarray) -> np.ndarray:
    """Zero mean, unit standard deviation."""
    return ((v - v.mean()) / (v.std() + 1e-12)).astype(np.float32)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))
print("🧰 toolkit loaded")

### Data: atoms of RuBisCO and density maps of the zoo

We reuse two data sources from Module 01: the atoms of **RuBisCO** (PDB 1BXN: the toolkit downloads its PDB file, or
writes one for a code-built stand-in when offline; we keep only the **Cα atoms**, one per amino acid, which trace the
protein backbone) and the density maps of four zoo molecules.

In [ ]:
# ── Data: Cα atoms of RuBisCO (PDB 1BXN; offline: of the code-built stand-in) ──
def read_ca_atoms(lines: list[str]) -> dict:
    """Cα atoms per chain from PDB ATOM records → {chain: (residue_numbers, xyz[n, 3] in Å)}. Keeps the first model,
    the first of alternative positions (column 17) and residues without an insertion code (column 27)."""
    chains = {}
    for line in lines:
        if line.startswith("ENDMDL"):
            break
        if line.startswith("ATOM  ") and line[12:16] == " CA " and line[16] in " A" and line[26] == " ":
            res, xyz = int(line[22:26]), [float(line[30:38]), float(line[38:46]), float(line[46:54])]
            chains.setdefault(line[21], ([], []))
            chains[line[21]][0].append(res)
            chains[line[21]][1].append(xyz)
    return {ch: (np.array(r), np.array(x)) for ch, (r, x) in chains.items()}


pdb_lines, ca_source = pdb_text_lines("1bxn")
ca = read_ca_atoms(pdb_lines)
print("Cα atoms of", ca_source)
for ch, (res, xyz_ch) in ca.items():
    print(f"  chain {ch}: {len(res):4d} Cα atoms, residues {res.min()}–{res.max()}")
LARGE = sorted(sorted(ca, key=lambda c: -len(ca[c][0]))[:4])       # the four longest chains = the large subunits
CH = LARGE[0]                                                       # the large subunit we study in Section 1
print("large-subunit chains:", LARGE)

In [ ]:
# ── Data: density maps of four zoo molecules (32³ boxes, 1 nm voxels) ──
CAST = ["4v4r", "3j9i", "6utj", "1bxn"]
zoo = load_zoo(CAST, size=32, sigma=1.0)

---
## 1 · Vectors and coordinates

> 📍 **Where we are.** Module 01 treated a map as a grid of numbers and built one from atoms. Now we treat positions as
> **vectors** — arrows from the origin to a point $(x, y, z)$ — because rotations act on vectors. Everything in this
> module (turning atoms, turning maps, comparing orientations) is built on three vector operations.

| Operation | Formula | Meaning |
|---|---|---|
| difference | $\mathbf d = \mathbf q - \mathbf p$ | the arrow from point p to point q |
| length (norm) | $\lVert\mathbf d\rVert = \sqrt{d_x^2 + d_y^2 + d_z^2}$ | the distance between p and q (Pythagoras in 3D) |
| dot product | $\mathbf a\cdot\mathbf b = a_xb_x + a_yb_y + a_zb_z = \lVert\mathbf a\rVert\,\lVert\mathbf b\rVert\cos\alpha$ | how aligned two arrows are; gives the angle α between them |

Let us test these on a protein whose geometry we know: along a polypeptide chain, consecutive Cα atoms are
**3.8 Å** apart (the peptide bond is planar and almost always *trans*). The angle between consecutive Cα→Cα arrows
depends on the secondary structure: about **90°** inside an α-helix (the chain turns sharply to wind the helix) and
about **120°** in an extended β-strand. RuBisCO's large subunit has both (its core is an α/β barrel) and loops in
between. As a clean reference we also build one ideal **β-strand** with the toolkit's backbone builder: it chains atoms
with ideal bond lengths and angles and sets the two backbone torsion angles of every residue (φ = −120°, ψ = +130° for
a strand; φ = −58°, ψ = −47° for a helix).

In [ ]:
# ── 1.1 Distances and angles along the backbone of a RuBisCO large subunit (+ one β-strand) ──
res_A, xyz_A = ca[CH]


def virtual_angles(xyz: np.ndarray) -> np.ndarray:
    """Cα–Cα–Cα angle (°) at every inner Cα of a continuous chain."""
    d = np.diff(xyz, axis=0)
    c = np.sum(d[:-1] * d[1:], axis=1) / (np.linalg.norm(d[:-1], axis=1) * np.linalg.norm(d[1:], axis=1))
    return 180 - np.degrees(np.arccos(np.clip(c, -1, 1)))


strand_ca = build_backbone(np.full(40, -120.0), np.full(40, 130.0))[:, 1]     # Cα atoms of a 40-residue β-strand
strand_angle = virtual_angles(strand_ca)

bonds = np.diff(xyz_A, axis=0)                       # arrows from each Cα to the next one
consecutive = np.diff(res_A) == 1                     # skip gaps where residues are missing
dist = np.linalg.norm(bonds, axis=1)[consecutive]     # lengths of the arrows
b1, b2 = bonds[:-1], bonds[1:]
ok = consecutive[:-1] & consecutive[1:]
cos_a = np.sum(b1 * b2, axis=1) / (np.linalg.norm(b1, axis=1) * np.linalg.norm(b2, axis=1))
virtual_angle = 180 - np.degrees(np.arccos(np.clip(cos_a[ok], -1, 1)))   # angle at the middle Cα

centre_A = xyz_A.mean(axis=0)                                           # centre (unweighted mean)
radius_of_gyration = np.sqrt(np.mean(np.sum((xyz_A - centre_A)**2, axis=1)))
print(f"median Cα–Cα distance: {np.median(dist):.2f} Å (n = {dist.size})")
print(f"centre of chain {CH}: {np.round(centre_A, 1)} Å;  radius of gyration: {radius_of_gyration:.1f} Å")
print(f"median Cα–Cα–Cα angle: chain {CH} {np.median(virtual_angle):.0f}°, β-strand {np.median(strand_angle):.0f}°")

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].hist(dist, bins=40, range=(3.0, 4.6), color="tab:blue")
axes[0].set_xlabel("distance between consecutive Cα atoms (Å)"); axes[0].set_ylabel("count")
axes[0].set_title("Bond-length check: one sharp peak")
axes[1].hist(virtual_angle, bins=60, range=(60, 160), color="tab:green", label=f"chain {CH} (large subunit)")
axes[1].hist(strand_angle, bins=60, range=(60, 160), color="tab:orange", label="built β-strand")
for a, lab in [(91, "α-helix"), (120, "β-strand")]:
    axes[1].axvline(a, color="k", ls="--", lw=1)
    axes[1].text(a + 1, axes[1].get_ylim()[1] * 0.85, lab, fontsize=8)
axes[1].set_xlabel("Cα–Cα–Cα angle (°)"); axes[1].set_ylabel("count")
axes[1].set_title("Angle check: two populations"); axes[1].legend(fontsize=7, loc="center right")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With nothing but differences, lengths and dot products we recovered textbook protein
geometry from raw coordinates: a 3.8 Å spacing and two angle families (helix and strand). In the real protein both
families are broad (and loops fill the space between them) because torsion angles vary from residue to residue; only
the ideal strand we built gives one sharp line. (Offline, the stand-in is made of ideal helices joined by straight
linkers, whose 180° angles fall off the plot.) The **radius of gyration**
(root-mean-square distance of the atoms from their centre) is a one-number summary of size used, for example, in
small-angle X-ray scattering.

---
## 2 · Rotations in 2D: derive the matrix from the unit circle

Rotating a point by an angle θ (counter-clockwise) around the origin keeps its distance from the origin and changes its
direction. Where do the two basic arrows go?

- $\mathbf e_x = (1, 0)$ lands on the unit circle at angle θ: $(\cos\theta, \sin\theta)$.
- $\mathbf e_y = (0, 1)$ starts 90° further, so it lands at angle θ + 90°: $(-\sin\theta, \cos\theta)$.

Any point is a mix of these two arrows, $(x, y) = x\,\mathbf e_x + y\,\mathbf e_y$, so it lands at
$x(\cos\theta, \sin\theta) + y(-\sin\theta, \cos\theta)$. Written as a matrix (the columns are where $\mathbf e_x$ and
$\mathbf e_y$ go):

$$R(\theta) = \begin{pmatrix}\cos\theta & -\sin\theta\\ \sin\theta & \cos\theta\end{pmatrix},\qquad
\begin{pmatrix}x'\\y'\end{pmatrix} = R(\theta)\begin{pmatrix}x\\y\end{pmatrix}$$

*In words:* the new x is "x·cos − y·sin", the new y is "x·sin + y·cos". Three properties define a rotation:
**lengths are preserved**, the inverse is the **transpose** ($R^\top R = I$; rotating back by −θ), and the
**determinant is +1** (a determinant of −1 would mean a mirror reflection).

In [ ]:
# ── 2.1 The 2D rotation matrix and its properties ──
def rot2d(theta: float) -> np.ndarray:
    """2×2 matrix rotating (x, y) vectors counter-clockwise by theta (radians)."""
    return np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])


R30 = rot2d(np.radians(30))
print("R(30°) =\n", np.round(R30, 3))
print("RᵀR =\n", np.round(R30.T @ R30, 10))
print("det R =", round(np.linalg.det(R30), 10))
print("R(30°)·R(45°) equals R(75°):", np.allclose(R30 @ rot2d(np.radians(45)), rot2d(np.radians(75))))
pts2d = xyz_A[:, :2] - xyz_A[:, :2].mean(axis=0)               # the large subunit seen down z, centred
rotated = pts2d @ R30.T                                        # rotate every point (rows) at once
pair_d = lambda p: np.linalg.norm(p[:, None] - p[None], axis=-1)   # all pairwise distances
print("all pairwise distances unchanged:", np.allclose(pair_d(pts2d[:200]), pair_d(rotated[:200])))

💡 `pts @ R.T` rotates many points stored as **rows** at once (because $(R\mathbf p)^\top = \mathbf p^\top R^\top$) —
a pattern you will see in almost every geometry code.

Rotating an *image* is different from rotating points: for every pixel of the output we must ask "which input pixel
lands here?" and **interpolate** between pixels (Section 6). The toolkit's `rotate_image` does that. ⚠️ `imshow`
draws row 0 at the top, so y points *down* on screen and a counter-clockwise rotation looks clockwise. We use
`origin="lower"` below so that y points up, as in maths.

In [ ]:
# ── 2.2 Rotating arrows, atoms and an image by 60° ──
theta = np.radians(60)
R60 = rot2d(theta)
ribo_proj = zoo["4v4r"].sum(axis=0)                            # a ribosome "micrograph" (projection along z)
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4))
circle = np.linspace(0, 2 * np.pi, 200)
axes[0].plot(np.cos(circle), np.sin(circle), color="0.7")
for v, col, lab in [((1, 0), "tab:blue", "e_x"), ((0, 1), "tab:red", "e_y")]:
    w = R60 @ np.array(v)
    axes[0].annotate("", xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=col, lw=1.5, alpha=0.4))
    axes[0].annotate("", xy=w, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=col, lw=2))
    axes[0].text(*(1.12 * w), f"R{lab}", color=col, fontsize=9, ha="center")
axes[0].set_aspect("equal"); axes[0].set_xlim(-1.3, 1.3); axes[0].set_ylim(-1.3, 1.3)
axes[0].set_title("columns of R = where e_x, e_y go (60°)")
axes[1].scatter(*pts2d.T, s=2, label=f"chain {CH} (Cα)")
axes[1].scatter(*(pts2d @ R60.T).T, s=2, label="rotated 60°")
axes[1].set_aspect("equal"); axes[1].legend(fontsize=7, markerscale=4)
axes[1].set_xlabel("x (Å)"); axes[1].set_ylabel("y (Å)"); axes[1].set_title("rotating atoms: pts @ R.T")
axes[2].imshow(rotate_image(ribo_proj, theta), origin="lower", cmap="Greys")
axes[2].contour(ribo_proj, levels=[ribo_proj.max() * 0.4], colors="tab:blue", origin="lower")
axes[2].contour(rotate_image(ribo_proj, theta), levels=[ribo_proj.max() * 0.4], colors="tab:orange", origin="lower")
axes[2].set_xlabel("x (nm)"); axes[2].set_ylabel("y (nm)")
axes[2].set_title("image outline (blue) → rotated 60° (orange)")
plt.tight_layout(); plt.show()

---
## 3 · Rotations in 3D

In 3D a rotation turns space about an **axis**. Rotating about the z-axis leaves z unchanged and turns (x, y) exactly
like the 2D matrix; similarly for x and y (the "right-hand rule": point your right thumb along the axis, your fingers
curl in the positive direction):

$$R_x(\alpha)=\begin{pmatrix}1&0&0\\0&\cos\alpha&-\sin\alpha\\0&\sin\alpha&\cos\alpha\end{pmatrix},\;
R_y(\alpha)=\begin{pmatrix}\cos\alpha&0&\sin\alpha\\0&1&0\\-\sin\alpha&0&\cos\alpha\end{pmatrix},\;
R_z(\alpha)=\begin{pmatrix}\cos\alpha&-\sin\alpha&0\\\sin\alpha&\cos\alpha&0\\0&0&1\end{pmatrix}$$

For a rotation by angle ω about **any** unit axis $\mathbf a$, **Rodrigues' formula** gives
$R = I + \sin\omega\,[\mathbf a]_\times + (1-\cos\omega)\,[\mathbf a]_\times^2$, where $[\mathbf a]_\times$ is the
matrix that computes the cross product $\mathbf a \times \mathbf v$. *In words:* keep the part of a vector along the
axis, and turn the part perpendicular to it by ω. Conversely, every rotation matrix has an axis (the direction it
leaves unchanged: the eigenvector with eigenvalue 1) and an angle (from its trace: $\operatorname{tr}R = 1 + 2\cos\omega$).

**Order matters.** Applying $R_1$ then $R_2$ is the product $R_2R_1$ (read right to left). Unlike 2D, 3D rotations do
not commute. Try it with a book: turn it 90° about a vertical axis, then 90° about a horizontal axis — and compare with
the opposite order.

In [ ]:
# ── 3.1 Rotation about any axis (Rodrigues) and its inverse (axis + angle from a matrix) ──
def axis_angle_matrix(axis, angle: float) -> np.ndarray:
    """Rotation by `angle` (radians, right-hand rule) about the unit vector `axis` (x, y, z) — Rodrigues' formula."""
    a = np.asarray(axis, float) / np.linalg.norm(axis)
    K = np.array([[0, -a[2], a[1]], [a[2], 0, -a[0]], [-a[1], a[0], 0]])     # K @ v == np.cross(a, v)
    return np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * K @ K


def matrix_axis_angle(R: np.ndarray) -> tuple[np.ndarray, float]:
    """Axis (unit vector, x y z) and angle (degrees) of a rotation matrix."""
    w, V = np.linalg.eig(R)
    axis = np.real(V[:, np.argmin(np.abs(w - 1))])                          # eigenvector with eigenvalue 1
    angle = np.degrees(np.arccos(np.clip((np.trace(R) - 1) / 2, -1, 1)))
    if np.dot(axis, [R[2, 1] - R[1, 2], R[0, 2] - R[2, 0], R[1, 0] - R[0, 1]]) < 0:
        axis = -axis                                                         # choose the sign of a right-hand turn
    return axis, float(angle)


print("Rodrigues about z equals rot_z:", np.allclose(axis_angle_matrix([0, 0, 1], 0.7), rot_z(0.7)))
R_test = axis_angle_matrix([1, 2, 2], np.radians(40))
ax_, ang_ = matrix_axis_angle(R_test)
print("recovered axis", np.round(ax_, 3), "(expected", np.round(np.array([1, 2, 2]) / 3, 3), ") angle", round(ang_, 2), "°")

In [ ]:
# ── 3.2 Order matters: Rz(90°)·Rx(90°) versus Rx(90°)·Rz(90°) ──
q = np.pi / 2
R_a = rot_z(q) @ rot_x(q)          # first about x, then about z
R_b = rot_x(q) @ rot_z(q)          # first about z, then about x
print("Rz·Rx =\n", np.round(R_a).astype(int), "\nRx·Rz =\n", np.round(R_b).astype(int))
print("same rotation?", np.allclose(R_a, R_b), "| angle between them:", round(rotation_angle_deg(R_a, R_b), 1), "°")

ribo = zoo["4v4r"]
fig, axes = plt.subplots(1, 3, figsize=(8, 2.8))
for ax, (vol, title) in zip(axes, [(ribo, "original"), (rotate_volume(ribo, R_a), "x first, then z"),
                                    (rotate_volume(ribo, R_b), "z first, then x")]):
    ax.imshow(vol.sum(axis=0)); ax.set_title(f"ribosome: {title}", fontsize=9); ax.axis("off")
fig.suptitle("Two 90° turns in different orders give different views (projection along z)", fontsize=9)
plt.tight_layout(); plt.show()

---
## 4 · Euler angles: three numbers for one orientation

A rotation matrix has 9 entries but only **3 degrees of freedom** (think: two numbers to say where the molecule's
"nose" points, one to say how it is spun about that direction). Cryo-EM software stores orientations as three
**Euler angles**. This course (like AITom/TomoMiner) uses the **ZYZ** convention:

$$R(\varphi,\theta,\psi) = R_z(\varphi)\,R_y(\theta)\,R_z(\psi),\qquad \varphi,\psi\in[0,2\pi),\ \theta\in[0,\pi]$$

*In words* (read right to left): spin the molecule by ψ about z, tilt it by θ about y, then spin it by φ about z.
The pair (θ, φ) says where the molecule's original z-axis ends up; ψ is the spin around it.

⚠️ **Conventions differ** between programs (ZYZ vs ZXZ, rotating the object vs the coordinate frame, degrees vs
radians, sign of angles). Converting orientations between software is a classic source of errors — always check the
documentation and test with a known rotation.

**Gimbal lock.** When θ = 0, $R_y(0) = I$ and $R_z(\varphi)R_z(\psi) = R_z(\varphi+\psi)$: only the *sum* φ + ψ matters,
so infinitely many angle triples describe the same rotation, and near θ = 0 tiny changes of orientation can require huge
changes of φ and ψ. This is a property of the angle *coordinates*, not of rotations themselves — and a reason why
optimisers and neural networks that output angles must be designed with care.

In [ ]:
# ── 4.1 Euler angles ↔ matrix, and gimbal lock ──
angles = np.radians([30, 50, 120])
R_e = euler_zyz_to_matrix(*angles)
back = matrix_to_euler_zyz(R_e)
print("(φ, θ, ψ) =", np.round(np.degrees(angles), 1), "→ matrix → angles", np.round(np.degrees(back), 1))
print("round trip gives the same matrix:", np.allclose(euler_zyz_to_matrix(*back), R_e))
print("orthogonal:", np.allclose(R_e.T @ R_e, np.eye(3)), "| det:", round(np.linalg.det(R_e), 6))

d = np.radians(25)
R_g1 = euler_zyz_to_matrix(np.radians(40), 0.0, np.radians(70))
R_g2 = euler_zyz_to_matrix(np.radians(40) + d, 0.0, np.radians(70) - d)
print("\nGimbal lock: (40°, 0°, 70°) and (65°, 0°, 45°) are the same rotation:", np.allclose(R_g1, R_g2))
R_n1 = euler_zyz_to_matrix(np.radians(40), np.radians(0.5), np.radians(70))
R_n2 = euler_zyz_to_matrix(np.radians(220), np.radians(0.5), np.radians(-110))
print("near θ = 0: angle triples 180° apart in φ and ψ are only",
      round(rotation_angle_deg(R_n1, R_n2), 2), "° apart as rotations")

---
## 5 · Random orientations: uniform is not what you think

Simulations (Module 09), data augmentation (Module 05) and search grids (Module 12) need orientations drawn
**uniformly** — every orientation equally likely, like molecules tumbling freely. The tempting recipe "draw φ, θ, ψ
uniformly in their ranges" is **wrong**: it puts too many molecules near θ = 0 and θ = 180°, because a band of
latitude near the pole is a much smaller area than a band near the equator. Correct recipes:

1. draw $\cos\theta$ (not θ) uniformly in [−1, 1], and φ, ψ uniformly; or
2. draw a random **unit quaternion** — four numbers $(w, x, y, z)$ from a normal distribution, divided by their length —
   and convert it to a matrix (the toolkit's `random_rotation`). A unit quaternion
   $(\cos\frac{\omega}{2}, \sin\frac{\omega}{2}\,\mathbf a)$ encodes a rotation by ω about axis $\mathbf a$; a uniformly
   random point on the 4D sphere gives a uniformly random rotation.

Two checks of uniformity: (a) the rotated z-axis $R\mathbf e_z$ should cover the sphere evenly, i.e. its z-coordinate
should be uniform in [−1, 1]; (b) the rotation angle ω of a uniform random rotation has the known density
$p(\omega) = (1-\cos\omega)/\pi$ on [0°, 180°] — most random rotations are large.

In [ ]:
# ── 5.1 Naive Euler angles versus correct samplers ──
n_samples = 4000
naive = [euler_zyz_to_matrix(rng.uniform(0, 2 * np.pi), rng.uniform(0, np.pi), rng.uniform(0, 2 * np.pi))
         for _ in range(n_samples)]
fixed = [euler_zyz_to_matrix(rng.uniform(0, 2 * np.pi), np.arccos(rng.uniform(-1, 1)), rng.uniform(0, 2 * np.pi))
         for _ in range(n_samples)]
quat = [random_rotation(rng) for _ in range(n_samples)]
z_axis = {name: np.array([R[:, 2] for R in rots]) for name, rots in
          [("naive Euler", naive), ("cos θ uniform", fixed), ("quaternion", quat)]}
omega = np.array([rotation_angle_deg(np.eye(3), R) for R in quat])

fig = plt.figure(figsize=(12, 3.2))
for k, name in enumerate(["naive Euler", "quaternion"]):
    ax = fig.add_subplot(1, 4, k + 1, projection="3d")
    p = z_axis[name][:1500]
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], s=1, alpha=0.5, color="tab:red" if k == 0 else "tab:green")
    ax.set_title(f"R·e_z, {name}", fontsize=9); ax.set_box_aspect((1, 1, 1))
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
ax = fig.add_subplot(1, 4, 3)
for name, col in [("naive Euler", "tab:red"), ("cos θ uniform", "tab:blue"), ("quaternion", "tab:green")]:
    ax.hist(z_axis[name][:, 2], bins=20, range=(-1, 1), histtype="step", lw=1.5, color=col, label=name)
ax.set_xlabel("z-coordinate of R·e_z  (= cos θ)"); ax.set_ylabel("count"); ax.legend(fontsize=7)
ax.set_title("uniform ⇒ flat histogram")
ax = fig.add_subplot(1, 4, 4)
ax.hist(omega, bins=30, range=(0, 180), density=True, color="tab:green", alpha=0.6, label="quaternion sampler")
w = np.linspace(0, np.pi, 200)
ax.plot(np.degrees(w), (1 - np.cos(w)) / np.pi * np.pi / 180, "k", label="(1 − cos ω)/π")
ax.set_xlabel("rotation angle ω (°)"); ax.set_ylabel("density (1/°)"); ax.legend(fontsize=7)
ax.set_title("angles of random rotations")
plt.tight_layout(); plt.show()
print(f"median rotation angle of uniform random rotations: {np.median(omega):.0f}°")

🔍 **What just happened?** The naive sampler crowds the poles (red, U-shaped histogram); both correct samplers are
flat. And the angle histogram matches theory: a "random orientation" is typically a *large* turn (median ≈ 132° in
theory), which
is why aligning randomly oriented particles is hard.

### 5.2 A pitfall from real code: a "random rotation" that is a mirror

Another tempting recipe is to take an orthogonal factor of a random matrix $A$, e.g. $U$ or $V^\top$ from its
singular value decomposition (SVD) $A = U S V^\top$. Both are orthogonal ($U^\top U = I$), but nothing forces their
determinant to be +1. A determinant of **−1** means a rotation combined with a **reflection** (a mirror). In fact
$\det U \cdot \det V^\top = \operatorname{sign}(\det A)$ (because $\det S > 0$), and $\det A < 0$ for half of all random
matrices — so in those cases at least one of the two factors *must* be a reflection. *Which* one depends on the internal
sign conventions of the linear-algebra library, so the bug can be invisible on one computer and present on another.

Why it matters: molecules are **chiral** (L-amino acids, right-handed α-helices, right-handed DNA). A reflected ribosome
is a molecule that does not exist in nature. Let us count.

In [ ]:
# ── 5.2 Orthogonal factors of random matrices: how often are they reflections? ──
det_U, det_Vt, det_A = [], [], []
for _ in range(2000):
    A = rng.normal(size=(3, 3))
    U, _, Vt = np.linalg.svd(A)
    det_U.append(np.linalg.det(U)); det_Vt.append(np.linalg.det(Vt)); det_A.append(np.linalg.det(A))
det_U, det_Vt, det_A = np.sign(det_U), np.sign(det_Vt), np.sign(det_A)
print(f"U  is a reflection (det = −1) in {np.mean(det_U < 0):5.1%} of draws")
print(f"Vᵀ is a reflection (det = −1) in {np.mean(det_Vt < 0):5.1%} of draws")
print(f"det(U)·det(Vᵀ) = sign(det A) in every draw: {np.all(det_U * det_Vt == det_A)};  "
      f"det A < 0 in {np.mean(det_A < 0):.1%} of draws")


def fix_reflection(U: np.ndarray) -> np.ndarray:
    """Turn an orthogonal matrix into a proper rotation by flipping one column if det = −1."""
    return U if np.linalg.det(U) > 0 else U @ np.diag([1, 1, -1])


fixed_dets = [np.linalg.det(fix_reflection(factor)) for _ in range(200)
              for factor in np.linalg.svd(rng.normal(size=(3, 3)))[::2]]          # both U and Vᵀ
print("after fix_reflection, all determinants are +1:", bool(np.all(np.array(fixed_dets) > 0)))

### 📄 Paper connection — a reflection hidden in a data-augmentation helper (DISCA)

**Paper:** Zeng X, Kahng A, Xue L, Mahamid J, Chang Y-W, Xu M (2023). *High-throughput cryo-ET structural pattern mining
by unsupervised deep iterative subtomogram clustering.* PNAS 120(15):e2213149120.
[doi:10.1073/pnas.2213149120](https://doi.org/10.1073/pnas.2213149120) (Module 22 covers the method itself.)

- **Why it matters here:** DISCA trains a network on subtomograms augmented by random rotations. In the released AITom
  code (`aitom/classify/deep/unsupervised/disca/util.py`), `random_rotation_matrix()` returns the **U factor of an SVD of
  a random matrix** — orthogonal, but not guaranteed to have determinant +1, so depending on the library some "rotated"
  training copies could be mirror images. (The same code fills empty corners with noise only in one branch of its
  augmentation function.)
- **What to take away:** always check `det(R) = +1` for rotations, and use a proper sampler (quaternions) — the
  course's `random_rotation` does. This is a code-level observation from the lab's own repository, used here as a
  lesson in careful reading; it is not a claim about the paper's reported results.

---
## 6 · Rotating a density map: interpolation and its cost

To rotate a map we do not move voxels (they would land between grid points and leave holes). Instead, for every output
voxel at position $\mathbf p$ we ask where it **came from** and read the input there:

$$V_{\text{out}}(\mathbf p) = V_{\text{in}}\big(R^\top(\mathbf p - \mathbf c) + \mathbf c\big)$$

where $\mathbf c$ is the box centre. *In words:* "pull" each output value from the inversely rotated position. That
position is generally between voxels, so we **interpolate**: order 0 = nearest voxel, order 1 = (tri)linear average of
the 8 surrounding voxels, order 3 = smooth cubic spline. This is exactly what the toolkit's `rotate_volume` does (with
`scipy.ndimage.affine_transform`); it also takes care of the `[z, y, x]` vs (x, y, z) bookkeeping.

In [ ]:
# ── 6.1 Rotate the ribosome by a known rotation ──
R_known = euler_zyz_to_matrix(*np.radians([30, 60, 90]))
ribo_rot = rotate_volume(ribo, R_known, order=1)
fig, axes = plt.subplots(2, 3, figsize=(7, 4.6))
show_titles = ["xy (z mid)", "xz (y mid)", "yz (x mid)"]
for row, (vol, lab) in enumerate([(ribo, "original"), (ribo_rot, "rotated (30°, 60°, 90°)")]):
    c = np.array(vol.shape) // 2
    for col, im in enumerate([vol[c[0]], vol[:, c[1], :], vol[:, :, c[2]]]):
        axes[row, col].imshow(im); axes[row, col].axis("off")
        axes[row, col].set_title(f"{lab}\n{show_titles[col]}", fontsize=8)
plt.tight_layout(); plt.show()
back_rot = rotate_volume(ribo_rot, R_known.T, order=1)          # undo the rotation with the transpose
print(f"correlation original vs rotated: {pearson(ribo, ribo_rot):.3f};  "
      f"after rotating back: {pearson(ribo, back_rot):.4f}")

Every interpolation smooths a little. One rotation hardly matters, but **repeated** resampling accumulates blur. We
rotate the map 36 times by 10° about the same axis — a full turn — and compare with rotating it *once* by the combined
rotation, for the three interpolation orders.

In [ ]:
# ── 6.2 Interpolation blur accumulates with repeated resampling ──
R10 = axis_angle_matrix([1, 2, 3], np.radians(10))
t0 = time.time()
curves = {}
for order in [0, 1, 3]:
    v = ribo.copy()
    scores = []
    for k in range(1, 37):
        v = rotate_volume(v, R10, order=order)                      # resample again and again
        if k % 3 == 0:
            once = rotate_volume(ribo, np.linalg.matrix_power(R10, k), order=order)   # one resampling
            scores.append(pearson(v, once))
    curves[order] = scores
print(f"done in {time.time() - t0:.1f} s; after the full turn: " +
      ", ".join(f"order {o}: r = {s[-1]:.3f}" for o, s in curves.items()))

fig, ax = plt.subplots(figsize=(6.4, 3.0))
for order, col in [(0, "tab:red"), (1, "tab:blue"), (3, "tab:green")]:
    ax.plot(np.arange(3, 37, 3), curves[order], "o-", color=col, label=f"order {order}")
ax.set_xlabel("number of successive 10° rotations"); ax.set_ylabel("Pearson r vs single resampling")
ax.set_title("Compose rotations as matrices, resample once"); ax.legend(fontsize=8); plt.show()

🔍 **What just happened?** Nearest-neighbour (order 0) scrambles voxels; linear interpolation (order 1, our default for
speed) slowly blurs; cubic splines (order 3) are much better but slower. The practical rule used in every alignment
and averaging program: **multiply the rotation matrices first, then resample the data only once.** Note also that a
rotated box loses its corners (filled with `cval = 0`) — padding the box or filling the corners with noise avoids
artificial edges.

---
## 7 · Rigid transforms: rotation plus translation

A molecule that is turned *and* moved undergoes a **rigid transform** $\mathbf x' = R\mathbf x + \mathbf t$ (distances
between atoms are preserved). Its inverse is $\mathbf x = R^\top(\mathbf x' - \mathbf t)$. Programs often pack both
parts into one 4×4 **homogeneous matrix** $\begin{pmatrix}R & \mathbf t\\ \mathbf 0^\top & 1\end{pmatrix}$ acting on
$(x, y, z, 1)$, so that chaining transforms is just multiplying matrices (Gum-Net's spatial transformer, Module 21, uses
such a matrix).

**Superposition.** Given two copies of a structure with matched points $\mathbf p_i$ and $\mathbf q_i$, which rigid
transform best moves P onto Q (smallest root-mean-square deviation, **RMSD**)? The **Kabsch algorithm** answers it in
closed form: centre both sets, form the 3×3 matrix $H = \sum_i (\mathbf p_i - \bar{\mathbf p})(\mathbf q_i - \bar{\mathbf q})^\top$,
take its SVD $H = U S V^\top$, and set $R = V\,\mathrm{diag}(1, 1, d)\,U^\top$ with $d = \operatorname{sign}\det(VU^\top)$.
That little $d$ is the reflection guard from Section 5.2: without it, the "best rotation" could be a mirror.

RuBisCO's file contains four copies of the large subunit (the chains in `LARGE`). Let us superpose them.

In [ ]:
# ── 7.1 The Kabsch algorithm ──
def kabsch(P: np.ndarray, Q: np.ndarray, allow_reflection: bool = False):
    """Best rigid transform (R, t) moving points P onto Q (rows matched); returns (R, t, rmsd_Å)."""
    p0, q0 = P.mean(axis=0), Q.mean(axis=0)
    H = (P - p0).T @ (Q - q0)
    U, _, Vt = np.linalg.svd(H)
    d = 1.0 if allow_reflection else np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1, 1, d]) @ U.T
    t = q0 - R @ p0
    rmsd = float(np.sqrt(np.mean(np.sum((P @ R.T + t - Q)**2, axis=1))))
    return R, t, rmsd


def matched(ch1: str, ch2: str):
    """Cα coordinates of the residues present in both chains, in the same order."""
    (r1, x1), (r2, x2) = ca[ch1], ca[ch2]
    common = np.intersect1d(r1, r2)
    return x1[np.searchsorted(r1, common)], x2[np.searchsorted(r2, common)]


# a self-test with a known transform
R_true, t_true = random_rotation(rng), np.array([5.0, -3.0, 12.0])
P_test = xyz_A
R_fit, t_fit, rmsd_fit = kabsch(P_test, P_test @ R_true.T + t_true)
print(f"self-test: rotation error {rotation_angle_deg(R_fit, R_true):.2e}°, shift error "
      f"{np.linalg.norm(t_fit - t_true):.2e} Å, RMSD {rmsd_fit:.2e} Å")

for other in LARGE[1:]:
    P, Q = matched(CH, other)
    R_ac, t_ac, rmsd = kabsch(P, Q)
    axis, angle = matrix_axis_angle(R_ac)
    before = np.sqrt(np.mean(np.sum((P - Q)**2, axis=1)))
    print(f"{CH} → {other}: RMSD before {before:5.1f} Å, after {rmsd:5.2f} Å; rotation {angle:6.1f}° about "
          f"axis {np.round(axis, 3)}")

In [ ]:
# ── 7.2 Superposition of one large subunit onto another, and the mirror test ──
CH2 = LARGE[2]
P, Q = matched(CH, CH2)
R_ac, t_ac, rmsd_ac = kabsch(P, Q)
Q_mirror = Q * np.array([-1, 1, 1])                                       # mirror image of chain CH2 (x → −x)
_, _, rmsd_mirror = kabsch(P, Q_mirror)
_, _, rmsd_cheat = kabsch(P, Q_mirror, allow_reflection=True)
print(f"{CH} onto mirrored {CH2}: best proper rotation RMSD = {rmsd_mirror:.1f} Å; "
      f"if reflections are allowed: {rmsd_cheat:.2f} Å (a 'perfect' fit that is physically impossible)")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, (A_pts, title) in zip(axes, [(P, f"before (RMSD {np.sqrt(np.mean(np.sum((P - Q)**2, 1))):.0f} Å)"),
                                     (P @ R_ac.T + t_ac, f"after Kabsch (RMSD {rmsd_ac:.2f} Å)")]):
    ax.plot(Q[:, 0], Q[:, 1], lw=0.8, color="tab:orange", label=f"chain {CH2}")
    ax.plot(A_pts[:, 0], A_pts[:, 1], lw=0.8, color="tab:blue", alpha=0.7, label=f"chain {CH}")
    ax.set_aspect("equal"); ax.set_title(title, fontsize=9); ax.set_xlabel("x (Å)"); ax.set_ylabel("y (Å)")
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Kabsch found the rigid transform that lays one large subunit on another. The three
rotations are 90°, 90° and 180° (a 270° turn is a 90° turn the other way) about the **same axis**: the four copies in
this file are related by a **4-fold rotational symmetry** (the L₄S₄ half of RuBisCO is C4; the complete L₈S₈ enzyme
adds 2-fold axes perpendicular to it, D4). After superposition the RMSD drops from tens of ångströms to 0.01 Å: the
depositors refined the four copies as exact copies ("strict non-crystallographic symmetry"). In other structures,
copies of a chain typically differ by a few tenths of an ångström (different crystal contacts, flexible loops). Superposing onto a
*mirror image* leaves a large RMSD unless we cheat by allowing reflections — chirality is real, and the determinant check
is what protects us.

---
## 8 · Measuring rotation error

Alignment methods estimate a rotation $R_{\text{est}}$ that should equal the truth $R_{\text{gt}}$. Two error measures
appear in the lab's papers:

- **Geodesic angle** — the angle of the single rotation that turns one into the other:
  $$\omega = \arccos\!\Big(\frac{\operatorname{tr}(R_{\text{est}}^\top R_{\text{gt}}) - 1}{2}\Big)\in[0°, 180°].$$
  (toolkit `rotation_angle_deg`). Intuitive and in degrees.
- **Frobenius distance** — treat the matrices as 9 numbers and take the Euclidean distance:
  $$\lVert R_{\text{est}} - R_{\text{gt}}\rVert_F = \sqrt{\textstyle\sum_{ij}(R_{\text{est}} - R_{\text{gt}})_{ij}^2}.$$

They are tied by an exact formula. Because $\lVert A\rVert_F^2 = \operatorname{tr}(A^\top A)$ and $R^\top R = I$:
$\lVert R_1 - R_2\rVert_F^2 = 2\operatorname{tr}I - 2\operatorname{tr}(R_1^\top R_2) = 6 - 2(1 + 2\cos\omega) = 8\sin^2(\omega/2)$, so

$$\lVert R_1 - R_2\rVert_F = 2\sqrt2\,\sin(\omega/2).$$

*In words:* the Frobenius distance grows almost linearly with the angle for small errors (≈ 0.0247 per degree) and
saturates at $2\sqrt2 \approx 2.83$ for a 180° error. Also $\lVert R_1^\top R_2 - I\rVert_F = \lVert R_2 - R_1\rVert_F$.

In [ ]:
# ── 8.1 Frobenius distance versus geodesic angle ──
pairs = [(random_rotation(rng), random_rotation(rng)) for _ in range(400)]
small = [(R, R @ axis_angle_matrix(rng.normal(size=3), np.radians(rng.uniform(0, 20)))) for R, _ in pairs[:200]]
om_deg = np.array([rotation_angle_deg(a, b) for a, b in pairs + small])
frob = np.array([np.linalg.norm(a - b) for a, b in pairs + small])
w = np.linspace(0, 180, 200)

fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.scatter(om_deg, frob, s=6, alpha=0.5, label="random pairs of rotations")
ax.plot(w, 2 * np.sqrt(2) * np.sin(np.radians(w) / 2), "k", lw=1, label="2√2·sin(ω/2)")
ax.axhline(0.1, color="tab:red", ls="--", lw=1)
ax.text(100, 0.18, "ΔE = 0.1 (failure threshold of Xu et al. 2012)", color="tab:red", fontsize=8)
ax.set_xlabel("geodesic angle ω (°)"); ax.set_ylabel("Frobenius distance ‖R₁ − R₂‖_F")
ax.set_title("Two ways to measure rotation error"); ax.legend(fontsize=8); plt.show()
assert np.allclose(frob, 2 * np.sqrt(2) * np.sin(np.radians(om_deg) / 2), atol=1e-6)


def frobenius_to_degrees(f: float) -> float:
    """Geodesic angle (°) of a rotation error given as a Frobenius distance."""
    return float(np.degrees(2 * np.arcsin(np.clip(f / (2 * np.sqrt(2)), 0, 1))))


for f in [0.1, 0.5, 1.0, 1.5, 2.0]:
    print(f"Frobenius {f:4.2f}  ↔  {frobenius_to_degrees(f):6.1f}°")

### 📄 Paper connection — how the lab's alignment papers report rotation error

| Paper | Error measure (from the paper / code) | Note |
|---|---|---|
| Xu, Beck & Alber 2012, *J Struct Biol* 178:152 ([doi:10.1016/j.jsb.2012.02.014](https://doi.org/10.1016/j.jsb.2012.02.014)) | $\Delta E = \lVert R_{\text{true}}^\top R_{\text{align}} - I\rVert_F$; $\Delta E > 0.1$ counts as a failure | 0.1 ↔ about 4° (table above) — a strict criterion. Module 12. |
| Gum-Net, Zeng & Xu, CVPR 2020 | $\lVert R_{\text{est}} - R_{\text{gt}}\rVert_F$ (code `angle_zyz_difference`) | the authors report e.g. 1.50 for Gum-Net vs 2.38 for both classical baselines at SNR 0.01 (Table 1). Module 21. |
| BOE-ViT, Jiang *et al.*, CVPR 2025 | geodesic angle $\arccos((\operatorname{tr}(R_{\text{est}}^\top R_{\text{gt}}) - 1)/2)$ | see the critical-reading exercise below. Module 21. |

**Question it answers:** "how wrong is an estimated orientation?" — the basic metric of every alignment benchmark.
**What our toy keeps:** both formulas exactly, and their conversion. **What it leaves out:** symmetry (Section 9: for a
7-fold symmetric particle, rotations differing by 360°/7 about the axis are equally correct, so errors should be taken
as the minimum over symmetry-equivalent rotations) and averaging over many pairs (the mean of Frobenius distances is
not the Frobenius distance of the mean angle).

✏️ **Critical-reading exercise.** In the BOE-ViT supplement, Gum-Net's rotation error on one complex (PDB 1I6V) at
SNR 0.01 appears as 1.42 ± 0.78, and the extracted table label says "degrees". Use `frobenius_to_degrees` and the
conversion radians → degrees to decide what 1.42 would mean if it were (a) degrees, (b) radians, (c) a Frobenius
distance. Which readings are plausible, given that Gum-Net's own paper reports 1.50 (Frobenius) at SNR 0.01? *Our
research notes flag this unit as unresolved — check the table header in the paper before quoting it.*

In [ ]:
# ── ✏️ 8.2 Your turn: three readings of the number 1.42 ──
value = 1.42
print(f"as degrees:            {value:.2f}°")
print(f"as radians:            {np.degrees(value):.1f}°")
print(f"as Frobenius distance: {frobenius_to_degrees(value):.1f}°")

---
## 9 · Symmetry: finding the proteasome's 7-fold axis

🧫 The **20S proteasome** of *Thermoplasma acidophilum* (PDB 3J9I) is a barrel of four stacked rings, each made of
seven subunits (α₇β₇β₇α₇). The protein-cutting active sites face the inside of the barrel, so only unfolded chains
threaded through the narrow entrance get degraded. Its symmetry is **D7**: one 7-fold axis along the barrel plus seven
2-fold axes perpendicular to it. A **symmetry operation** is a rotation that leaves the density unchanged:
$V(R^\top\mathbf x) = V(\mathbf x)$ for all $\mathbf x$.

Symmetry is valuable: averaging the 7 (or 14) symmetry-related copies inside each particle boosts the signal, and the
symmetry must be accounted for when measuring alignment errors. Here we *discover* it with only rotate-and-correlate:

1. **Axis.** For many candidate axis directions $\mathbf a$, rotate the map by 360°/7 about $\mathbf a$ and compute the
   correlation with the original; the true axis gives r ≈ 1. As a sanity check, compare with the **principal axes** of
   the map (eigenvectors of its second-moment matrix — the long axis of an elongated barrel).
2. **Order.** A smooth barrel is *nearly* unchanged by **any** rotation about its axis, so the raw correlation barely
   reveals "7". We therefore remove the cylindrical average (the part that every rotation about the axis preserves) and
   correlate what remains — the bumps of the subunits — at every rotation angle.

In [ ]:
# ── 9.1 Candidate axes: principal axes and a scan over directions ──
prot_raw, _, prot_src = zoo_map("3j9i")
prot = prepare_map(prot_raw, size=32, sigma=0.5)                     # less blur keeps the subunit bumps
g = np.arange(32) - 15.5
gz, gy, gx = np.meshgrid(g, g, g, indexing="ij")
coords = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)      # (x, y, z) of every voxel
wts = prot.ravel() / prot.sum()
second_moment = (coords * wts[:, None]).T @ coords                     # 3×3 matrix of weighted x², xy, ...
evals, evecs = np.linalg.eigh(second_moment)
long_axis = evecs[:, np.argmax(evals)]


def fibonacci_hemisphere(n: int) -> np.ndarray:
    """n roughly evenly spaced unit vectors (x, y, z) on the upper hemisphere (an axis and its negative are equal)."""
    i = np.arange(n) + 0.5
    polar = np.arccos(1 - i / n)
    azim = np.pi * (1 + 5**0.5) * i
    return np.stack([np.cos(azim) * np.sin(polar), np.sin(azim) * np.sin(polar), np.cos(polar)], axis=1)


t0 = time.time()
directions = fibonacci_hemisphere(300)
corr_7 = np.array([pearson(prot, rotate_volume(prot, axis_angle_matrix(a, 2 * np.pi / 7))) for a in directions])
axis7 = directions[np.argmax(corr_7)]
print(f"scanned {len(directions)} axes in {time.time() - t0:.1f} s; best r = {corr_7.max():.4f} "
      f"(median over directions {np.median(corr_7):.3f})")
print(f"best axis {np.round(axis7, 3)}; angle to the long principal axis: "
      f"{np.degrees(np.arccos(abs(axis7 @ long_axis))):.1f}°")

fig, ax = plt.subplots(figsize=(6.4, 3.2))
pol_deg = np.degrees(np.arccos(directions[:, 2]))
az_deg = np.degrees(np.arctan2(directions[:, 1], directions[:, 0])) % 360
sc = ax.scatter(az_deg, pol_deg, c=corr_7, cmap="viridis", s=25)
la = long_axis * np.sign(long_axis[2] if long_axis[2] != 0 else 1)
ax.scatter(np.degrees(np.arctan2(la[1], la[0])) % 360, np.degrees(np.arccos(la[2])), marker="x", s=120,
           color="tab:red", label="long principal axis")
plt.colorbar(sc, ax=ax, label="r after a 360°/7 turn")
ax.set_xlabel("axis azimuth (°)"); ax.set_ylabel("axis polar angle (°)"); ax.legend(fontsize=8, loc="lower right")
ax.set_title("Which axis is the 7-fold axis? (one dot per candidate)"); plt.show()

In [ ]:
# ── 9.2 The order of the symmetry: correlate the non-cylindrical part at every angle ──
cyl = np.mean([rotate_volume(prot, axis_angle_matrix(axis7, a)) for a in np.linspace(0, 2 * np.pi, 36, endpoint=False)],
              axis=0)                                                   # cylindrical average about the axis
bumps = prot - cyl                                                      # what makes the barrel not a smooth tube
angles_deg = np.arange(0, 361, 3)
corr_angle = [pearson(bumps, rotate_volume(bumps, axis_angle_matrix(axis7, np.radians(a)))) for a in angles_deg]
orders = np.arange(2, 13)
corr_order = [pearson(bumps, rotate_volume(bumps, axis_angle_matrix(axis7, 2 * np.pi / n))) for n in orders]
best_order = int(orders[np.argmax(corr_order)])
print(f"best n-fold order: {best_order}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(angles_deg, corr_angle, color="tab:purple")
for k in range(1, 7):
    axes[0].axvline(360 * k / 7, color="0.6", ls=":", lw=1)
axes[0].set_xlabel("rotation about the axis (°)"); axes[0].set_ylabel("r (non-cylindrical part)")
axes[0].set_title("Peaks every 360°/7 ≈ 51.4° (dotted lines)")
axes[1].bar(orders, corr_order, color=["tab:purple" if n == best_order else "0.7" for n in orders])
axes[1].set_xlabel("candidate order n (turn by 360°/n)"); axes[1].set_ylabel("r")
axes[1].set_title("Which n-fold symmetry?")
plt.tight_layout(); plt.show()
assert best_order == 7, "the 20S proteasome should show 7-fold symmetry"

In [ ]:
# ── 9.3 Turn the barrel so that its symmetry axis points along z, and look down the axis ──
to_z = axis_angle_matrix(np.cross(axis7, [0, 0, 1]), np.arccos(np.clip(axis7 @ [0, 0, 1], -1, 1)))
upright = rotate_volume(prot, to_z, order=3)
inner = upright[13:19].mean(axis=0)                     # the two inner (β) rings, averaged over 6 slices
c0, r_ring = 15.5, 4.5                                  # box centre and the radius of the subunit ring (voxels = nm)
phi = np.radians(np.arange(0, 360, 2))
profile = ndimage.map_coordinates(inner, [c0 + r_ring * np.sin(phi), c0 + r_ring * np.cos(phi)], order=1)

fig = plt.figure(figsize=(10.5, 2.8))
ax1, ax2, ax3 = fig.add_subplot(1, 4, 1), fig.add_subplot(1, 4, 2), fig.add_subplot(1, 4, 3)
ax1.imshow(upright.sum(axis=1)); ax1.set_title("side view", fontsize=9)
ax2.imshow(upright.sum(axis=0)); ax2.set_title("projection down the axis", fontsize=9)
ax3.imshow(inner); ax3.set_title("inner rings (slice average)", fontsize=9)
ax3.plot(c0 + r_ring * np.cos(phi), c0 + r_ring * np.sin(phi), color="tab:orange", lw=1)
for ax in (ax1, ax2, ax3):
    ax.axis("off")
ax4 = fig.add_subplot(1, 4, 4)
ax4.plot(np.degrees(phi), profile, color="tab:orange")
for k in range(7):
    ax4.axvline(np.degrees(phi[np.argmax(profile)]) % (360 / 7) + 360 * k / 7, color="0.6", ls=":", lw=1)
ax4.set_xlabel("angle around the ring (°)"); ax4.set_ylabel("density on the orange circle", fontsize=8)
ax4.set_title("seven subunits, 51.4° apart", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Rotate-and-correlate found the barrel axis (it coincides with the long principal axis) and
the hidden 7-fold periodicity. Turned upright, the barrel looks almost circular at 1 nm resolution, but reading the
density along a circle through the subunits (orange) shows seven bumps — the seven subunits of each ring. The same "rotate and compare"
logic, run over *all* rotations and *two* different maps, is subtomogram alignment (Module 12).

✏️ **Your turn.** D7 symmetry also has seven **2-fold** axes perpendicular to the barrel axis. The cell below rotates
the upright map by 180° about horizontal axes at different in-plane angles. How many peaks do you see between 0° and
180°, and how far apart are they? *(Expected: 7 peaks, 360°/14 ≈ 25.7° apart. If the peaks are weak, the map may not
be perfectly centred on its symmetry centre — try `order=3` or a finer angle step.)*

In [ ]:
# ── ✏️ 9.4 Your turn: look for the 2-fold axes of D7 ──
inplane = np.arange(0, 180, 2.5)
corr_2fold = [pearson(upright, rotate_volume(upright, axis_angle_matrix([np.cos(np.radians(a)), np.sin(np.radians(a)), 0],
                                                                         np.pi))) for a in inplane]
fig, ax = plt.subplots(figsize=(6.4, 2.6))
ax.plot(inplane, corr_2fold, "o-", ms=3)
ax.set_xlabel("direction of the horizontal axis (°)"); ax.set_ylabel("r after a 180° turn")
ax.set_title("Searching for 2-fold axes perpendicular to the barrel"); plt.show()

---
## 10 · Rotation-invariant features: describing a molecule without knowing its orientation

Alignment searches over rotations — expensive (Module 12). A cheaper idea for *recognising* or *grouping* molecules:
describe each density by numbers that **do not change when the molecule turns**. Such **rotation-invariant features**
can be compared directly. This is the idea of Xu, Zhang & Alber (2009), later used voxel-by-voxel for template-free
detection (Xu, Beck & Alber 2011; Module 15).

**Step 1 — shells.** Cut space around the centre into thin concentric spheres (shells) of radius $r_1 < r_2 < \dots$.
A rotation about the centre maps each shell onto itself; it only moves the pattern *on* the shell.

**Step 2 — the simplest invariant.** The mean density on each shell (the **radial profile**) cannot change under
rotation. But it throws away *where* on the shell the density sits: a ring and a disc of the same mass at the same
radius look identical to it.

**Step 3 — spherical harmonics.** Just as a periodic 1D signal is a sum of sines and cosines of increasing frequency
(Module 03), any pattern $g(\theta, \phi)$ on a sphere is a sum of **spherical harmonics** $Y_l^m(\theta,\phi)$:

$$g(\theta,\phi) = \sum_{l=0}^{\infty}\ \sum_{m=-l}^{l} a_{lm}\,Y_l^m(\theta,\phi),\qquad
a_{lm} = \int g\,\overline{Y_l^m}\,d\Omega$$

Here θ is the polar angle (0 at the north pole), φ the azimuth (longitude), and the **degree** $l$ is an angular
frequency: $l = 0$ is constant, $l = 1$ is "one side vs the other", $l = 2$ an elongation, larger $l$ finer
patterns (a 7-fold ring needs $l \ge 7$). Each degree has $2l + 1$ functions ($m = -l..l$).

**Step 4 — the invariant.** Rotating the pattern **mixes the coefficients $a_{lm}$ with the same $l$ among themselves**
(by a unitary matrix, the Wigner D-matrix) but never mixes different $l$, and never changes their total length. So the
**energy per degree** on shell $j$

$$s_{jl} = \Big(\sum_{m=-l}^{l} |a^{(j)}_{lm}|^2\Big)^{1/2}$$

is rotation-invariant. *In words:* turning a globe moves the continents, but not how much "wiggliness" of each angular
frequency the map contains. The feature vector of a molecule is the table $s_{jl}$ over shells $j$ and degrees $l$.

🧮 **Computing $a_{lm}$.** We sample each shell on a grid of polar angles (Gauss–Legendre points in cos θ) and azimuths
(evenly spaced), read the density there by interpolation, and replace the integral by a weighted sum. SciPy provides
$Y_l^m$ as `scipy.special.sph_harm_y(l, m, θ_polar, φ_azimuth)` (SciPy ≥ 1.15); older versions had
`sph_harm(m, l, φ_azimuth, θ_polar)` — note the swapped argument order — which was deprecated and later removed. The
next cell works with either.

In [ ]:
# ── 10.1 Spherical harmonics (version-proof) and what they look like ──
try:
    from scipy.special import sph_harm_y as _sph_harm_y           # SciPy ≥ 1.15

    def Y_lm(l: int, m: int, polar, azimuth):
        """Complex spherical harmonic Y_l^m (orthonormal on the unit sphere); angles in radians."""
        return _sph_harm_y(l, m, polar, azimuth)
    SH_SOURCE = "scipy.special.sph_harm_y"
except ImportError:                                                 # older SciPy
    from scipy.special import sph_harm as _sph_harm

    def Y_lm(l: int, m: int, polar, azimuth):
        """Complex spherical harmonic Y_l^m (orthonormal on the unit sphere); angles in radians."""
        return _sph_harm(m, l, azimuth, polar)
    SH_SOURCE = "scipy.special.sph_harm (legacy argument order)"
print("using", SH_SOURCE)

pol_grid, az_grid = np.meshgrid(np.linspace(0, np.pi, 60), np.linspace(0, 2 * np.pi, 120), indexing="ij")
examples = [(0, 0), (1, 0), (1, 1), (2, 0), (2, 2), (3, 2), (7, 7), (8, 4)]
fig, axes = plt.subplots(2, 4, figsize=(10, 3.6))
for ax, (l, m) in zip(axes.ravel(), examples):
    vals = np.real(Y_lm(l, m, pol_grid, az_grid))
    lim = np.abs(vals).max()
    ax.imshow(vals, cmap="RdBu_r", vmin=-lim, vmax=lim, extent=(0, 360, 180, 0), aspect="auto")
    ax.set_title(f"Re Y_{l}^{m}  (l = {l})", fontsize=9)
    ax.set_xticks([0, 180, 360]); ax.set_yticks([0, 90, 180]); ax.tick_params(labelsize=7)
for ax in axes[1]:
    ax.set_xlabel("azimuth φ (°)", fontsize=8)
for ax in axes[:, 0]:
    ax.set_ylabel("polar θ (°)", fontsize=8)
fig.suptitle("Spherical harmonics drawn as world maps (red +, blue −): l counts the angular wiggles", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# ── 10.2 A spherical-harmonic shell descriptor ──
def build_sh_descriptor(box: int, radii, L: int = 8) -> dict:
    """Precompute sampling points, quadrature weights and Y_l^m values for shells of the given radii (voxels)."""
    nodes, gl_w = np.polynomial.legendre.leggauss(2 * (L + 1))           # Gauss–Legendre nodes in cos(θ)
    n_az = 4 * (L + 1)
    polar, azim = np.meshgrid(np.arccos(nodes), np.arange(n_az) * 2 * np.pi / n_az, indexing="ij")
    weights = (gl_w[:, None] * np.full(n_az, 2 * np.pi / n_az)[None, :]).ravel()   # ∫ dΩ ≈ Σ weights
    polar, azim = polar.ravel(), azim.ravel()
    lm = [(l, m) for l in range(L + 1) for m in range(-l, l + 1)]
    Y = np.array([Y_lm(l, m, polar, azim) for l, m in lm]).T              # (points, n_lm)
    Yw = np.conj(Y) * weights[:, None]                                    # a_lm = Σ_points f · conj(Y) · weight
    gram_error = np.abs(Yw.T @ Y - np.eye(len(lm))).max()                 # orthonormality check of the quadrature
    unit_zyx = np.stack([np.cos(polar), np.sin(polar) * np.sin(azim), np.sin(polar) * np.cos(azim)], axis=1)
    centre = (box - 1) / 2
    points = np.concatenate([centre + r * unit_zyx for r in radii]).T    # [z, y, x] sampling coordinates
    return dict(L=L, radii=np.asarray(radii), points=points, Yw=Yw, degree=np.array([l for l, _ in lm]),
                n_per_shell=len(polar), gram_error=gram_error)


def sh_shell_features(vol: np.ndarray, desc: dict) -> np.ndarray:
    """Rotation-invariant energies s[j, l] = sqrt(Σ_m |a_lm|²) of the density on each shell j (shape: shells × (L+1))."""
    f = ndimage.map_coordinates(vol, desc["points"], order=1).reshape(len(desc["radii"]), desc["n_per_shell"])
    power = np.abs(f @ desc["Yw"])**2                                     # |a_lm|² for every shell
    return np.sqrt(np.stack([power[:, desc["degree"] == l].sum(axis=1) for l in range(desc["L"] + 1)], axis=1))


SH = build_sh_descriptor(box=32, radii=np.arange(1, 15), L=8)
print(f"descriptor: {len(SH['radii'])} shells × {SH['L'] + 1} degrees = {len(SH['radii']) * (SH['L'] + 1)} numbers; "
      f"{SH['n_per_shell']} sample points per shell")
print(f"quadrature check — the sampled Y_lm are orthonormal to within {SH['gram_error']:.1e}")
assert SH["gram_error"] < 1e-8

Now the key test: rotate the ribosome by several random rotations and compare (a) the voxel-by-voxel correlation of
the rotated maps with the original — which drops, because voxels moved — and (b) the shell features — which should
stay the same up to small interpolation errors.

In [ ]:
# ── 10.3 Invariance test ──
F_ref = sh_shell_features(ribo, SH)
rows = []
for k in range(5):
    R_k = random_rotation(rng)
    v_k = rotate_volume(ribo, R_k)
    F_k = sh_shell_features(v_k, SH)
    rows.append((rotation_angle_deg(np.eye(3), R_k), pearson(ribo, v_k),
                 np.linalg.norm(F_k - F_ref) / np.linalg.norm(F_ref)))
for ang, r_vox, rel in rows:
    print(f"rotation {ang:5.1f}°: voxel correlation r = {r_vox:.3f};  feature change = {100 * rel:.1f} %")
F_prot = sh_shell_features(zoo["3j9i"], SH)
print(f"for comparison, ribosome vs 20S proteasome features differ by "
      f"{100 * np.linalg.norm(F_prot - F_ref) / np.linalg.norm(F_ref):.0f} %")
assert max(r[2] for r in rows) < 0.05, "shell features should be (nearly) rotation-invariant"

fig, axes = plt.subplots(1, 4, figsize=(11, 2.8))
for ax, (M, title) in zip(axes, [(F_ref, "ribosome"), (F_k, "ribosome, randomly rotated"),
                                 (10 * np.abs(F_k - F_ref), "|difference| × 10"), (F_prot, "20S proteasome")]):
    im = ax.imshow(M, aspect="auto", cmap="viridis", vmin=0, vmax=F_ref.max(),
                   extent=(-0.5, SH["L"] + 0.5, SH["radii"][-1] + 0.5, SH["radii"][0] - 0.5))
    ax.set_title(title, fontsize=9); ax.set_xlabel("degree l"); ax.set_xticks(range(0, SH["L"] + 1, 2))
axes[0].set_ylabel("shell radius (nm)")
plt.colorbar(im, ax=axes[-1], label="s_jl")
plt.tight_layout(); plt.show()

🔍 **What just happened?** After a random turn, the voxel-wise correlation with the original falls well below 1 — the
maps are no longer superimposed — but the shell-energy table changes by only about 1 % (interpolation error), while a
*different* molecule changes it far more. The energy is concentrated at low degrees and mid radii: most of a
molecule's shape is coarse. Symmetry leaves fingerprints too: the D7 proteasome has energy (almost) only at **even**
degrees. Group theory explains why — the 7-fold axis allows only $m = 0$ terms below $l = 7$, and for $m = 0$ the
perpendicular 2-fold axes flip the sign of odd-$l$ harmonics, so those must vanish. The features have a catch, though: they are invariant to rotations **about the box
centre** only, so the particle must be centred (try shifting it by 2 voxels with `ndimage.shift` and recompute).

### 10.4 Can invariant features tell molecules apart in noise?

The 2009 paper compared three families of rotation-invariant descriptors — spherical harmonics, **moments** of the
density, and intensity **histograms** — at several noise levels. We run a small experiment in that spirit with four
zoo molecules (70S ribosome, 20S proteasome, capped proteasome, RuBisCO), 40 randomly oriented copies each, noise at
several SNRs, and four descriptors:

| descriptor | numbers | what it keeps |
|---|---|---|
| histogram of voxel values (inside a sphere) | 16 | only the distribution of values |
| moments: total mass + the 3 eigenvalues of the second-moment matrix | 4 | size and elongation |
| radial profile ($s_{j0}$ only) | 14 | how density is distributed with radius |
| full SH shell energies $s_{jl}$ | 126 | radius **and** angular structure |

Classification is deliberately simple — **nearest centroid**: standardise each feature, average the training copies of
each class, assign each test copy to the closest class average — repeated over three random train/test splits.

⚠️ **A fairness detail.** In a real tomogram the noise level is set by the electron dose, the same for every molecule.
So we add noise of the *same* strength to all classes (SNR defined relative to the average signal variance of the four
molecules), then normalise each copy to zero mean and unit variance. If instead each class got noise scaled to its own
variance, a classifier could "cheat" by measuring the noise level.

In [ ]:
# ── 10.4 Rotation-invariant descriptors versus noise ──
inside = (gz**2 + gy**2 + gx**2) <= 14**2                               # a sphere of radius 14 voxels
xyz_inside = coords[inside.ravel()]


def moment_features(vol: np.ndarray) -> np.ndarray:
    """Total (positive) mass and the 3 eigenvalues of the mass-weighted second-moment matrix inside the sphere."""
    w_ = np.clip(vol[inside], 0, None)
    m0 = w_.sum() + 1e-12
    com = (xyz_inside * w_[:, None]).sum(axis=0) / m0
    d_ = xyz_inside - com
    return np.concatenate([[m0], np.linalg.eigvalsh((d_ * w_[:, None]).T @ d_ / m0)])


def nearest_centroid_accuracy(F: np.ndarray, y: np.ndarray, rng, n_splits: int = 3) -> float:
    """Mean test accuracy of a nearest-centroid classifier over random half/half splits (features standardised)."""
    accs = []
    for _ in range(n_splits):
        idx = rng.permutation(len(y))
        tr, te = idx[:len(y) // 2], idx[len(y) // 2:]
        mu, sd = F[tr].mean(axis=0), F[tr].std(axis=0) + 1e-9
        Z = (F - mu) / sd
        cents = np.array([Z[tr][y[tr] == c_].mean(axis=0) for c_ in np.unique(y)])
        pred = np.argmin(((Z[te][:, None, :] - cents[None]) ** 2).sum(axis=-1), axis=1)
        accs.append(np.mean(pred == y[te]))
    return float(np.mean(accs))


t0 = time.time()
n_copies = 40
oriented = {pid: [rotate_volume(zoo[pid], random_rotation(rng)) for _ in range(n_copies)] for pid in CAST}
labels = np.repeat(np.arange(len(CAST)), n_copies)
signal_var = np.mean([zoo[pid].var() for pid in CAST])
snr_levels = [0.1, 0.03, 0.01, 0.005]
results = {name: [] for name in ["histogram", "moments", "radial profile", "SH shell energies"]}
features_at = {}
for snr in snr_levels:
    sigma = np.sqrt(signal_var / snr)                                  # the same noise strength for every molecule
    feats = {name: [] for name in results}
    for pid in CAST:
        for v in oriented[pid]:
            noisy = normalize(v + rng.normal(0, sigma, v.shape))
            S = sh_shell_features(noisy, SH)
            feats["SH shell energies"].append(S.ravel())
            feats["radial profile"].append(S[:, 0])
            feats["moments"].append(moment_features(noisy))
            feats["histogram"].append(np.histogram(noisy[inside], bins=np.linspace(-3, 3, 17))[0])
    for name in results:
        results[name].append(nearest_centroid_accuracy(np.array(feats[name], float), labels, rng))
    features_at[snr] = np.array(feats["SH shell energies"])
print(f"feature extraction for {len(labels) * len(snr_levels)} noisy volumes took {time.time() - t0:.1f} s")
for name, acc in results.items():
    print(f"  {name:18s}: " + "  ".join(f"SNR {s:<5}: {a:.2f}" for s, a in zip(snr_levels, acc)))

In [ ]:
# ── 10.5 Plot: accuracy versus SNR, and a 2D map of the SH features ──
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
for (name, acc), mk in zip(results.items(), ["s", "^", "o", "D"]):
    axes[0].plot(snr_levels, acc, marker=mk, label=name)
axes[0].axhline(1 / len(CAST), color="0.5", ls=":", lw=1)
axes[0].text(0.0055, 1 / len(CAST) + 0.02, "chance", fontsize=8, color="0.4")
axes[0].set_xscale("log"); axes[0].invert_xaxis()
axes[0].set_xlabel("SNR (log scale; noisier →)"); axes[0].set_ylabel("test accuracy (4 classes)")
axes[0].set_title("Rotation-invariant descriptors in noise"); axes[0].legend(fontsize=8)

snr_show = 0.03
F = features_at[snr_show]
Z = (F - F.mean(axis=0)) / (F.std(axis=0) + 1e-9)
U_, S_, Vt_ = np.linalg.svd(Z, full_matrices=False)                     # principal component analysis via SVD
pcs = Z @ Vt_[:2].T
for c_, pid in enumerate(CAST):
    axes[1].scatter(*pcs[labels == c_].T, s=12, label=ZOO[pid]["name"])
axes[1].set_xlabel("principal component 1"); axes[1].set_ylabel("principal component 2")
axes[1].set_title(f"SH features of randomly rotated copies (SNR {snr_show})"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Without ever aligning a single pair, invariant features grouped randomly oriented, noisy
copies by molecule (right: one cloud per molecule in a 2D summary of the 126 features, made with principal component
analysis — PCA, Module 04). The histogram — which knows nothing about shape — is the least accurate at every noise
level, while the spherical-harmonic energies, which keep both radial and angular structure, remain among the most
accurate and match or beat the radial profile (which is just their $l = 0$ column) at every noise level: angular
information helps. In our toy the four molecules differ strongly in size, so even the four "moment" numbers do as well
(at the highest noise, slightly better than the SH energies); in the PCA map, the 20S and the
capped proteasome — which differ only by the cap — lie closest together. Toys like this depend on their details (which molecules, how much noise,
box size), so treat the ranking as an illustration, not a replication.

✏️ **Your turn.** (1) Change `L=8` to `L=3` in the `build_sh_descriptor` call of 10.2 and re-run 10.2–10.5: do
higher angular frequencies help? (2) Replace `CAST` in 10.4 by `["3j9i", "6utj", "1qvr", "3cf3"]` (two proteasomes
and two ring-shaped AAA+ ATPases, all of similar size) — `load_zoo` will fetch the extra maps if you add them to the
`zoo` dictionary first: `zoo.update(load_zoo(["1qvr", "3cf3"]))`. Which descriptors survive when size no longer
gives the answer away?

### 📄 Paper connection — rotation-invariant features of density maps

**Paper:** Xu M, Zhang S, Alber F (2009). *3D Rotation Invariant Features for the Characterization of Molecular Density
Maps.* 2009 IEEE International Conference on Bioinformatics and Biomedicine (BIBM), pp. 74–78.
[doi:10.1109/BIBM.2009.32](https://doi.org/10.1109/BIBM.2009.32)

> ⚠️ We could not access the full text of this paper while writing; the description below is limited to its title and
> abstract (plus how the same descriptor is described in the lab's 2011 paper). Read the paper before relying on details.

- **Question it answers:** how can we tell that two local regions of density maps look alike when they may be arbitrarily
  rotated relative to each other and noisy — without a template and without searching over rotations?
- **What the paper does (abstract level):** it builds and compares three kinds of rotation-invariant local descriptors —
  a **spherical-harmonics** decomposition of the local density neighbourhood, **moments** of the local density, and
  intensity **histograms** — proposes FFT-based computation for speed, and evaluates them in a classification of protein
  density maps at several noise levels.
- **The descriptor as used later:** in Xu, Beck & Alber (2011, *Bioinformatics* 27:i69; Module 15) every voxel's
  neighbourhood is split into M concentric shells, the density on each shell is expanded in spherical harmonics, and
  each shell is summarised by L rotation-invariant numbers $s_{jl}$, concatenated over shells. The energy-per-degree
  formula $s_{jl} = (\sum_m |a_{lm}|^2)^{1/2}$ that we implemented is the standard invariant; our research notes could not
  confirm from the extracted text that it is exactly the papers' formula.
- **What our toy keeps:** concentric shells, SH expansion per shell, per-degree energies, and a noise-robustness
  comparison against moment and histogram descriptors.
- **What our toy leaves out:** local (per-voxel) features over a whole tomogram, FFT-based computation, the paper's
  datasets and classifier.
- **Result reported:** the abstract reports that at low noise all three descriptor types perform about equally, and at
  high noise the spherical-harmonics features are clearly more reliable than the moment- and histogram-based ones.
- **Code:** none in AITom. Related: spherical harmonics on *Fourier* shells power the fast rotational search of
  Xu, Beck & Alber (2012) — Module 12.

---
## ✅ Checkpoint

1. Write the 2D rotation matrix for θ = 90° and apply it to the point (2, 0).
   <details><summary>Answer</summary>R = [[0, −1], [1, 0]]; R·(2, 0) = (0, 2): a quarter turn counter-clockwise.</details>
2. Why is "draw θ uniformly in [0°, 180°]" a bad way to sample random orientations?
   <details><summary>Answer</summary>Equal steps of θ near the poles cover much less area of the sphere than near the
   equator, so orientations pile up at the poles. Draw cos θ uniformly in [−1, 1] (or use random unit quaternions).</details>
3. A method reports a rotation error of 0.1 as a Frobenius distance. About how many degrees is that? And 2.83?
   <details><summary>Answer</summary>ω = 2·arcsin(0.1 / 2√2) ≈ 4.1°. A Frobenius distance of 2√2 ≈ 2.83 is the maximum,
   a 180° error.</details>
4. An orthogonal matrix has determinant −1. What does it do to a ribosome, and why is that a problem?
   <details><summary>Answer</summary>It rotates *and mirrors* it, producing the opposite-handed (enantiomeric) molecule,
   which does not exist in nature (L-amino acids, right-handed helices). Training on or aligning to mirrored copies
   corrupts results.</details>
5. Why must you combine rotation matrices first and resample a map only once?
   <details><summary>Answer</summary>Every resampling interpolates and slightly blurs (or scrambles) the map; repeated
   resampling accumulates the damage, while multiplying matrices is exact.</details>
6. Why is $s_{jl} = (\sum_m |a_{lm}|^2)^{1/2}$ unchanged by a rotation, and what does it *not* capture?
   <details><summary>Answer</summary>A rotation mixes the a_lm of the same degree l by a unitary matrix, which preserves
   their total squared length. It does not capture how patterns on *different* shells (or different degrees) are
   oriented relative to each other, and it is not invariant to translations (the particle must be centred).</details>

## 🧠 Questions & discussion

1. The 20S proteasome has D7 symmetry. If you align a proteasome copy to a reference and report the rotation error
   without accounting for symmetry, what errors could you wrongly report? How would you fix the metric?
2. Invariant features discard orientation. When is that an advantage (classification, discovery) and when a
   disadvantage (averaging, locating a ligand-binding site)?
3. Rotation-invariant features compare *shapes* without alignment. Could two different molecules have identical
   feature tables? (Hint: what information does $s_{jl}$ throw away?)
4. Neural networks that regress Euler angles (e.g. Gum-Net) must cope with gimbal lock and with angles wrapping around
   at 360°. What alternative outputs would avoid these problems?
5. Why might an elongated barrel (proteasome) be easier to align about one axis than about another?

## 🚀 Bonus — going further

- **Quaternion algebra:** implement quaternion multiplication and verify that multiplying quaternions corresponds to
  multiplying rotation matrices; build a uniform rotation grid for a search (Module 12).
- **Symmetry-aware error:** write `symmetric_rotation_error(R_est, R_gt, axis, n)` = minimum geodesic angle over the n
  rotations about the symmetry axis; test it on 6UTJ (C7).
- **Local invariant features:** compute the SH descriptor around *every* voxel of a small crowded volume (Module 00's
  `make_crowded_cell`) and cluster voxels — the first step of Xu, Beck & Alber 2011 (Module 15).
- **Deep dive:** [spatial transformations tutorial](../../../../sci_img/tutorial/foundation/ComputationalSignalBase/SpatialTransformations/)
  in this repository (affine transforms, interpolation, homogeneous coordinates).
- **Real code:** read AITom's `aitom/geometry/` helpers and its rotation conventions; compare with our ZYZ functions.

## 📝 Lab note

```text
Module 02 — date:
Input: Cα atoms of RuBisCO (PDB 1BXN); zoo maps 4V4R, 3J9I, 6UTJ, 1BXN painted from PDB models (32³, 1 nm voxels)
Output: rotated atoms/maps, Kabsch superposition, error conversions, 7-fold axis, SH shell features, accuracy vs SNR
Method: rotation matrices (x/y/z, Rodrigues, ZYZ Euler), quaternion sampling, affine resampling, Kabsch (SVD + det
        guard), Frobenius/geodesic error, rotate-and-correlate symmetry search, SH energies per shell
Key parameter changed & effect (e.g. interpolation order, L, SNR):
Failure mode (e.g. reflection, gimbal lock, uncentred particle):
Paper link: Xu, Zhang & Alber 2009; Xu 2012 ΔE; Gum-Net / BOE-ViT metrics; DISCA rotation helper
```

## 📚 References

1. Xu M, Zhang S, Alber F. 3D rotation invariant features for the characterization of molecular density maps.
   *IEEE BIBM 2009*, pp. 74–78. doi:10.1109/BIBM.2009.32
2. Xu M, Beck M, Alber F. Template-free detection of macromolecular complexes in cryo electron tomograms.
   *Bioinformatics* 27(13):i69–i76 (2011). doi:10.1093/bioinformatics/btr207
3. Xu M, Beck M, Alber F. High-throughput subtomogram alignment and classification by Fourier space constrained fast
   volumetric matching. *J Struct Biol* 178(2):152–164 (2012). doi:10.1016/j.jsb.2012.02.014
4. Zeng X, Xu M. Gum-Net: Unsupervised geometric matching for fast and accurate 3D subtomogram image alignment and
   averaging. *CVPR 2020*, pp. 4073–4084.
5. Jiang R, Daggett J, Pingulkar S, Zhao Y, Dhingra P, Brown D, Wu Q, Zeng X, Li X, Xu M. BOE-ViT: Boosting orientation
   estimation with equivariance in self-supervised 3D subtomogram alignment. *CVPR 2025*, pp. 29352–29362.
6. Zeng X, Kahng A, Xue L, Mahamid J, Chang Y-W, Xu M. High-throughput cryo-ET structural pattern mining by unsupervised
   deep iterative subtomogram clustering. *PNAS* 120(15):e2213149120 (2023). doi:10.1073/pnas.2213149120
7. Kabsch W. A solution for the best rotation to relate two sets of vectors. *Acta Cryst A* 32:922–923 (1976).
   doi:10.1107/S0567739476001873
8. Shoemake K. Uniform random rotations. In *Graphics Gems III*, pp. 124–132, Academic Press (1992).

## ➡️ Next

**[03 · Waves and frequencies: the Fourier transform for biologists](03_fourier_frequency.ipynb)** — resolution is a
statement about frequencies. Next you will decompose images into waves, filter them, find shifts with the FFT, and see
why Fourier *magnitudes* ignore translations (the trick behind fast rotational search). Course overview:
[`README.md`](../README.md) · terms: [`GLOSSARY.md`](../GLOSSARY.md).